# Notebook 02 — Descarga de registros de precipitación

**TFM:** Evaluación de la influencia del componente espacial en la estimación de datos faltantes de precipitación diaria mediante modelos de Machine Learning aplicados a estaciones meteorológicas.

## Objetivo del notebook

Este Notebook parte de **estaciones candidatas** filtradas a partir de un **punto de interés** en el *Notebook 1*, a partir de estas estaciones se descargará la precipitación correspindientes.

## Flujo

1. Cargar el archivo de Salida del Notebook 1.
2. Crear la consulta SoQL para Socrata.
3. Estandarización de la información.
4. Disponibilidad Temrporal.
5. Guardar archivos intermedios para los siguientes notebooks.

# Obtener → preparar → caracterizar disponibilidad.

# instalación de Librerias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

# Detectar carpetas del proyecto

---
> ℹ️ **Información**
>
> A continuación se construye la estructura de archivos a manejar en el proyecto

In [ ]:
from pathlib import Path

carpeta_actual = Path.cwd()

if carpeta_actual.name == 'notebooks':
    PROJECT_ROOT = carpeta_actual.parent
else:
    PROJECT_ROOT = carpeta_actual

folders = [
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "intermediate",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "output" / "tables",
    PROJECT_ROOT / "output" / "figures",
    PROJECT_ROOT / "scr"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
# Archivo generado por el Notebook 01
ESTACIONES_CANDIDATAS_PATH =  PROJECT_ROOT / "data" / "intermediate" / "estaciones_candidatas.geojson"


In [ ]:
gdf_estaciones_candidatas = gpd.read_file(ESTACIONES_CANDIDATAS_PATH)
gdf_estaciones_candidatas.head()

,codigo,nombre,categoria,tecnologia,estado,departamento,municipio,altitud,longitud,latitud,...,area_operativa,area_hidrografica,zona_hidrografica,subzona_hidrografica,entidad,corriente,fecha_suspension,distancia_m,fecha_instalacion_mod,geometry
0,2120000100,COLEGIO ALEMANIA SOLIDARIA [2120000100],Pluviométrica,Automática con Telemetría,Activa,Bogotá,"Bogotá, D.C",2556,-74.075434,4.652389,...,Area Operativa 11 - Cundinamarca-Amazonas,Magdalena Cauca,Alto Magdalena,Río Bogotá,INSTITUTO DISTRITAL DE GESTIÓN DE RIESGOS Y CA...,None,None,1293.417942,2015-10-01,POINT (4880763.562 2072174.63)
1,0021205540,INST GEOFI ANDES [21205540],Climatológica Ordinaria,Convencional,Activa,Bogotá,"Bogotá, D.C",2600,-74.066667,4.633333,...,Area Operativa 11 - Cundinamarca-Amazonas,Magdalena Cauca,Alto Magdalena,Río Bogotá,ESTACIONES PARTICULARES,None,None,3573.971940,1942-01-15,POINT (4881732.559 2070067.274)
2,2120000170,COLEGIO SIMON RODRIGUEZ SEDE C [2120000170],Pluviométrica,Automática con Telemetría,Activa,Bogotá,"Bogotá, D.C",2702,-74.054350,4.640500,...,Area Operativa 11 - Cundinamarca-Amazonas,Magdalena Cauca,Alto Magdalena,Río Bogotá,INSTITUTO DISTRITAL DE GESTIÓN DE RIESGOS Y CA...,None,None,3605.128787,2019-10-01,POINT (4883099.511 2070857.232)
3,2120000135,COLEGIO MANUEL ELKIN PATARROYO [2120000135],Pluviométrica,Automática con Telemetría,Activa,Bogotá,"Bogotá, D.C",2672,-74.064390,4.618230,...,Area Operativa 11 - Cundinamarca-Amazonas,Magdalena Cauca,Alto Magdalena,Río Bogotá,INSTITUTO DISTRITAL DE GESTIÓN DE RIESGOS Y CA...,None,None,5244.923986,2018-10-01,POINT (4881982.515 2068397.796)
4,0021205290,EDIFICIO SAMPER [21205290],Climatológica Ordinaria,Convencional,Suspendida,Bogotá,"Bogotá, D.C",2570,-74.083333,4.616667,...,Area Operativa 11 - Cundinamarca-Amazonas,Magdalena Cauca,Alto Magdalena,Río Bogotá,MINISTERIO DE AGRICULTURA,None,15/01/1950,5279.355342,1944-01-15,POINT (4879881.606 2068228.198)


In [ ]:
# Dataset de estaciones del IDEAM
DATASET_ID = "s54a-sgyg"
URL_API = f"https://www.datos.gov.co/resource/{DATASET_ID}.json"


In [ ]:
codigos_estaciones_candidatas = gdf_estaciones_candidatas["codigo"].to_list()


codigos_soql = ", ".join(
    f"'{codigo}'" for codigo in codigos_estaciones_candidatas
)

codigos_soql

"'2120000100', '0021205540', '2120000170', '2120000135', '0021205290', '0021200230', '0021206890', '2120000120', '2120000139', '0021205380', '0021201120', '0021205520'"

In [ ]:
consulta_soql = f"""
SELECT
    codigoestacion,
    min(fechaobservacion) AS FechaInicio,
    max(fechaobservacion) AS FechaFinal
WHERE
  codigoestacion = {codigos_soql}
GROUP BY
  codigoestacion
"""

print(consulta_soql)



SELECT
    codigoestacion,
    min(fechaobservacion) AS FechaInicio,
    max(fechaobservacion) AS FechaFinal
WHERE
  codigoestacion in ('2120000100', '0021205540', '2120000170', '2120000135', '0021205290', '0021200230', '0021206890', '2120000120', '2120000139', '0021205380', '0021201120', '0021205520')
GROUP BY
  codigoestacion



In [ ]:

import requests

parametros = {
    "$query": consulta_soql
}

respuesta = requests.get(
    URL_API,
    params=parametros,
    timeout=120 # Increased timeout to 120 seconds
)

print("Código HTTP:", respuesta.status_code)
#print("Respuesta de la API:", respuesta.text)
print("Consulta enviada:")
print(consulta_soql)

respuesta.raise_for_status()

ReadTimeout: HTTPSConnectionPool(host='www.datos.gov.co', port=443): Read timed out. (read timeout=180)



---
prueba


In [ ]:
estacion_prueba = codigos_estaciones_candidatas[0]

In [ ]:
FECHA_INICIO = "1996-01-01T00:00:00"
FECHA_FIN = "2026-01-01T00:00:00"

In [ ]:
consulta_prueba = f"""
SELECT
    codigoestacion,
    fechaobservacion,
    valorobservado
WHERE
    codigoestacion = '{estacion_prueba}'
    AND fechaobservacion >= '2025-01-01T00:00:00'
    AND fechaobservacion < '2025-02-01T00:00:00'
LIMIT 5
"""

consulta_2 = f"""
SELECT
    codigoestacion,
    fechaobservacion
WHERE
    codigoestacion = '{estacion_prueba}'
    AND fechaobservacion >= '{FECHA_INICIO}'
    AND fechaobservacion < '{FECHA_FIN}'
ORDER BY
    fechaobservacion DESC
LIMIT 1
"""

In [ ]:
parametros = {
    "$query": consulta_prueba
}

respuesta = requests.get(
    URL_API,
    params=parametros,
    timeout=120
)

print("Código HTTP:", respuesta.status_code)
#print("Respuesta de la API:", respuesta.text)
print("Consulta enviada:")
print(consulta_soql)

respuesta.raise_for_status()

Código HTTP: 200
Consulta enviada:

SELECT
    codigoestacion,
    min(fechaobservacion) AS FechaInicio,
    max(fechaobservacion) AS FechaFinal
WHERE
  codigoestacion in ('2120000100', '0021205540', '2120000170', '2120000135', '0021205290', '0021200230', '0021206890', '2120000120', '2120000139', '0021205380', '0021201120', '0021205520')
GROUP BY
  codigoestacion



In [ ]:
datos = respuesta.json()
datos

[{'codigoestacion': '2120000100',
  'fechaobservacion': '2025-01-01T00:00:00.000',
  'valorobservado': '0'},
 {'codigoestacion': '2120000100',
  'fechaobservacion': '2025-01-01T00:00:00.000',
  'valorobservado': '0'},
 {'codigoestacion': '2120000100',
  'fechaobservacion': '2025-01-01T00:03:00.000',
  'valorobservado': '0'},
 {'codigoestacion': '2120000100',
  'fechaobservacion': '2025-01-01T00:03:00.000',
  'valorobservado': '0'},
 {'codigoestacion': '2120000100',
  'fechaobservacion': '2025-01-01T00:04:00.000',
  'valorobservado': '0'}]

In [ ]:
def importar_temporalidad(listado_codigos_estaciones):

    respuesta_estaciones = []

    for estacion in listado_codigos_estaciones:

        consulta_codEstacion_Soql = f"""
        SELECT
            codigoestacion,
            min(fechaobservacion) AS fecha_inicial,
            max(fechaobservacion) AS fecha_final

        WHERE
            codigoestacion = '{estacion}'

        GROUP BY
            codigoestacion
        """

        parametros = {
            "$query": consulta_codEstacion_Soql
        }

        respuesta = requests.get(
            URL_API,
            params=parametros,
            timeout=120
        )

        respuesta_estaciones.append(respuesta.json())

    return respuesta_estaciones

In [ ]:
def importar_temporalidad(listado_codigos_estaciones):

    respuesta_estaciones = []

    for estacion in listado_codigos_estaciones:

        consulta_codEstacion_Soql = f"""
        SELECT
            codigoestacion,
            min(fechaobservacion) AS fecha_inicial,
            max(fechaobservacion) AS fecha_final

        WHERE
            codigoestacion = '{estacion}'

        GROUP BY
            codigoestacion
        """

        parametros = {
            "$query": consulta_codEstacion_Soql
        }

        respuesta = requests.get(
            URL_API,
            params=parametros,
            timeout=120
        )

        respuesta_estaciones.append(respuesta.json())

    return respuesta_estaciones

In [ ]:
temporalidad_estaciones = importar_temporalidad(codigos_estaciones_candidatas)

ReadTimeout: HTTPSConnectionPool(host='www.datos.gov.co', port=443): Read timed out. (read timeout=120)

In [ ]:
print(temporalidad_estaciones)

None


In [ ]:
respuestaAPI_temporalidad_estaciones = respuesta.json()
periodos_registros_estaciones = pd.DataFrame(respuestaAPI_temporalidad_estaciones)
periodos_registros_estaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   CodigoEstacion  6 non-null      object
 1   FechaInicio     6 non-null      object
 2   FechaFinal      6 non-null      object
dtypes: object(3)
memory usage: 276.0+ bytes


In [ ]:
periodos_registros_estaciones

[{'CodigoEstacion': '0021206890',
  'FechaInicio': '2012-06-14T23:45:00.000',
  'FechaFinal': '2026-07-22T22:02:00.000'},
 {'CodigoEstacion': '2120000100',
  'FechaInicio': '2018-09-11T15:16:00.000',
  'FechaFinal': '2026-07-22T21:51:00.000'},
 {'CodigoEstacion': '2120000120',
  'FechaInicio': '2023-10-12T11:45:00.000',
  'FechaFinal': '2026-07-22T22:02:00.000'},
 {'CodigoEstacion': '2120000135',
  'FechaInicio': '2023-10-12T11:45:00.000',
  'FechaFinal': '2026-07-22T22:01:00.000'},
 {'CodigoEstacion': '2120000139',
  'FechaInicio': '2023-10-12T11:43:00.000',
  'FechaFinal': '2026-07-22T21:57:00.000'},
 {'CodigoEstacion': '2120000170',
  'FechaInicio': '2023-10-12T11:43:00.000',
  'FechaFinal': '2026-07-22T22:01:00.000'}]

In [ ]:
nombres_estaciones_candidatas = gdf_estaciones_candidatas["nombre"].to_list()


codigos_soql_2 = ", ".join(
    f"'{nombre}'" for nombre in nombres_estaciones_candidatas
)

codigos_soql_2

"'COLEGIO ALEMANIA SOLIDARIA [2120000100]', 'INST GEOFI ANDES  [21205540]', 'COLEGIO SIMON RODRIGUEZ SEDE C [2120000170]', 'COLEGIO MANUEL ELKIN PATARROYO [2120000135]', 'EDIFICIO SAMPER  [21205290]', 'SAN DIEGO  [21200230]', 'CERRO CAZADORES [21206890]', 'COLEGIO GUSTAVO MORALES [2120000120]', 'ESC. PEDAGOGICA EXPRIMENTAL [2120000139]', 'HIPOTECHO  [21205380]', 'CASITA LA  [21201120]', 'ELDORADO DIDACTICA [21205520]'"

In [ ]:
consulta_soql_2 = f"""
SELECT
    NombreEstacion,
    min(FechaObservacion) AS FechaInicio,
    max(FechaObservacion) AS FechaFinal
WHERE
  NombreEstacion in ({codigos_soql_2})
GROUP BY
  NombreEstacion
"""

print(consulta_soql_2)



SELECT
    NombreEstacion,
    min(FechaObservacion) AS FechaInicio,
    max(FechaObservacion) AS FechaFinal
WHERE
  NombreEstacion in ('COLEGIO ALEMANIA SOLIDARIA [2120000100]', 'INST GEOFI ANDES  [21205540]', 'COLEGIO SIMON RODRIGUEZ SEDE C [2120000170]', 'COLEGIO MANUEL ELKIN PATARROYO [2120000135]', 'EDIFICIO SAMPER  [21205290]', 'SAN DIEGO  [21200230]', 'CERRO CAZADORES [21206890]', 'COLEGIO GUSTAVO MORALES [2120000120]', 'ESC. PEDAGOGICA EXPRIMENTAL [2120000139]', 'HIPOTECHO  [21205380]', 'CASITA LA  [21201120]', 'ELDORADO DIDACTICA [21205520]')
GROUP BY
  NombreEstacion



In [ ]:
parametros_2 = {
    "$query": consulta_soql_2
}

respuesta_2 = requests.get(
    URL_API,
    params=parametros_2, # Corrected parameter to parametros_2
    timeout=120 # Increased timeout to 120 seconds
)

print("Código HTTP:", respuesta_2.status_code)
#print("Respuesta de la API:", respuesta_2.text)
print("Consulta enviada:")
print(consulta_soql_2)

respuesta_2.raise_for_status()

Código HTTP: 200
Consulta enviada:

SELECT
    NombreEstacion,
    min(FechaObservacion) AS FechaInicio,
    max(FechaObservacion) AS FechaFinal
WHERE
  NombreEstacion in ('COLEGIO ALEMANIA SOLIDARIA [2120000100]', 'INST GEOFI ANDES  [21205540]', 'COLEGIO SIMON RODRIGUEZ SEDE C [2120000170]', 'COLEGIO MANUEL ELKIN PATARROYO [2120000135]', 'EDIFICIO SAMPER  [21205290]', 'SAN DIEGO  [21200230]', 'CERRO CAZADORES [21206890]', 'COLEGIO GUSTAVO MORALES [2120000120]', 'ESC. PEDAGOGICA EXPRIMENTAL [2120000139]', 'HIPOTECHO  [21205380]', 'CASITA LA  [21201120]', 'ELDORADO DIDACTICA [21205520]')
GROUP BY
  NombreEstacion



In [ ]:
respuestaAPI_temporalidad_estaciones2 = respuesta_2.json()
respuestaAPI_temporalidad_estaciones2


[]

In [ ]:
from datetime import date

fecha_actual = date.today()
anio_final = fecha_actual.year - 1
anio_inicio = anio_final - 30

anios_para_consulta = []

for anio in range(anio_inicio, anio_final , 1):
    anios_para_consulta.append(anio)


In [ ]:
def importar_precipitacion(listado_codigos_estaciones, anios):

    respuesta_estaciones = []

    LIMIT = 20000

    for estacion in listado_codigos_estaciones:

        for anio in anios:

            FECHA_INICIO = f"{anio}-01-01T00:00:00"
            FECHA_FIN = f"{anio+1}-01-01T00:00:00"

            OFFSET = 0

            while True:

                consulta_anio_codestacion_soql = f"""
                SELECT
                    *
                WHERE
                    codigoestacion = '{estacion}'
                    AND fechaobservacion >= '{FECHA_INICIO}'
                    AND fechaobservacion < '{FECHA_FIN}'
                ORDER BY
                    fechaobservacion ASC,
                    :id ASC
                LIMIT {LIMIT}
                OFFSET {OFFSET}
                """

                parametros = {
                    "$query": consulta_anio_codestacion_soql
                }

                respuesta = requests.get(
                    URL_API,
                    params=parametros,
                    timeout=120
                )

                respuesta.raise_for_status()

                datos = respuesta.json()

                respuesta_estaciones.extend(datos)

                if len(datos) < LIMIT:
                    break

                OFFSET += LIMIT

    df = pd.DataFrame(respuesta_estaciones)

    return df

